In [2]:
# SimpleDirectoryReader is dynamic, detects file type and uses appropriate reader
from llama_index.core import SimpleDirectoryReader, Document, VectorStoreIndex, Settings, PromptTemplate, StorageContext, KnowledgeGraphIndex
from llama_index.core.utilities.sql_wrapper import SQLDatabase
from llama_index.core.query_engine import NLSQLTableQueryEngine, KnowledgeGraphQueryEngine
from llama_index.core.workflow import Workflow, StartEvent, StopEvent, step, Context, Event
from llama_index.core.retrievers import SQLRetriever
from llama_index.core.node_parser import SentenceSplitter
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.llms.openai import OpenAI
from llama_parse import LlamaParse

from transformers import AutoTokenizer

import pandas as pd, re, ast, textwrap
from sqlalchemy import create_engine

import openai
from neo4j import GraphDatabase
from llama_index.graph_stores.neo4j import Neo4jGraphStore

from dotenv import load_dotenv, find_dotenv
import torch
import os
import re
import json

# --- Azure Sandpit Environment ---
# Project root path for Azure Sandpit environment
# project_root_path = "/home/azureuser/cloudfiles/code/Users/TAN_Heng_Joo/heng-joo-capstone" 

# # Change the current working directory to the project root
# os.chdir(project_root_path)


# # --- FIX 2: Bypass find_dotenv() and use a direct, verified path ---
# dotenv_path = "/home/azureuser/cloudfiles/code/Users/TAN_Heng_Joo/heng-joo-capstone/.env"

# # Add a critical check to ensure the .env file exists at this path
# if not os.path.exists(dotenv_path):
#     raise FileNotFoundError(
#         f"CRITICAL ERROR: .env file NOT FOUND at the expected path: {dotenv_path}\n"
#         f"Please double-check the path you pasted into 'project_root_path'."
#     )

dotenv_path = find_dotenv()

# 2. Load the .env file from the path that was found.
load_dotenv(dotenv_path=dotenv_path)

# 3. Get the project root from the directory where the .env file was found.
project_root = os.path.dirname(dotenv_path)

# 4. Get the relative directory name from the environment variable
relative_data_dir = os.getenv("GRAPH_DATASET_DIR")

# 5. *** THIS IS THE CRITICAL FIX ***
#    Create the full, absolute path by joining the project root with the relative name.
data_directory = os.path.join(project_root, relative_data_dir)

hf_token = os.getenv("HUGGINGFACE_TOKEN")
llama_cloud_api_key = os.getenv("LLAMA_CLOUD_API_KEY")
openai_api_key = os.getenv("OPENAI_API_KEY")

# Neo4j Credentials
URI = os.getenv("NEO4J_URI")
AUTH = (os.getenv("NEO4J_DATABASE"), os.getenv("NEO4J_PASSWORD"))

# Check neo4j connectivity
with GraphDatabase.driver(URI, auth=AUTH) as driver:
    # Ensure the connection is valid
    driver.verify_connectivity()
    print("Connection successful!")


print(f"✅ Project root automatically determined as: {project_root}")
print(f"✅ .env file loaded from: {dotenv_path}")
print(f"📁 Data directory set to: {data_directory}")

Connection successful!
✅ Project root automatically determined as: /Users/joo/heng-joo-capstone
✅ .env file loaded from: /Users/joo/heng-joo-capstone/.env
📁 Data directory set to: /Users/joo/heng-joo-capstone/Datasets/Graph_Dataset


## Vector/Graph Store Data Ingestion

In [3]:
# Check if directory exists
if not data_directory or not os.path.isdir(data_directory):
    raise ValueError(
        f"The path '{data_directory}' is not a valid directory. "
        "Please check that the VECTOR_DATASET_DIR variable is set correctly in your .env file "
        "and that the directory actually exists."
    )

# Initialize LlamaParse with your API key
llama_cloud_api_key = os.getenv("LLAMA_CLOUD_API_KEY")
if not llama_cloud_api_key:
    raise ValueError("LLAMA_CLOUD_API_KEY not found in your .env file. Please get a key from https://cloud.llamaindex.ai")

parser = LlamaParse(
    api_key=llama_cloud_api_key,
    result_type="markdown",
    verbose=True
)

# Separate the file paths based on their type (PDF vs. other)
pdf_filepaths = []
other_filepaths = []
for filename in os.listdir(data_directory):
    file_path = os.path.join(data_directory, filename)
    if os.path.isfile(file_path):
        if filename.lower().endswith('.pdf'):
            pdf_filepaths.append(file_path)
        else:
            other_filepaths.append(file_path)

print(f"--- Found {len(pdf_filepaths)} PDF(s) and {len(other_filepaths)} other file(s) to process. ---")

# Process the files in batches
all_documents = []

# Process all PDFs in a single batch call to LlamaParse
if pdf_filepaths:
    print("\n- Parsing PDF files with LlamaParse...")
    try:
        # Calling parser.load_data() with a LIST of files is the correct way
        pdf_docs = parser.load_data(pdf_filepaths)
        all_documents.extend(pdf_docs)
        print(f"  -> Successfully parsed {len(pdf_filepaths)} PDF file(s).")
    except Exception as e:
        print(f"  -> FAILED to parse PDFs with LlamaParse. Error: {e}")

# Process all other files in a single batch call to SimpleDirectoryReader
if other_filepaths:
    print("\n- Parsing other files with SimpleDirectoryReader...")
    try:
        other_docs = SimpleDirectoryReader(input_files=other_filepaths).load_data()
        all_documents.extend(other_docs)
        print(f"  -> Successfully parsed {len(other_filepaths)} other file(s).")
    except Exception as e:
        print(f"  -> FAILED to parse other files. Error: {e}")

# The 'documents' variable should now contain all chunks from all parsed files
documents = all_documents
print(f"\n--- Ingestion complete ---")
print(f"Successfully loaded and chunked a total of {len(documents)} document(s) from all files in '{data_directory}'.")


--- Found 2 PDF(s) and 2 other file(s) to process. ---

- Parsing PDF files with LlamaParse...


Parsing files:   0%|          | 0/2 [00:00<?, ?it/s]Retrying llama_cloud_services.parse.utils.make_api_request.<locals>._make_request in 4.0 seconds as it raised ConnectError: [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate in certificate chain (_ssl.c:1028).
Retrying llama_cloud_services.parse.utils.make_api_request.<locals>._make_request in 4.0 seconds as it raised ConnectError: [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate in certificate chain (_ssl.c:1028).
Retrying llama_cloud_services.parse.utils.make_api_request.<locals>._make_request in 4.0 seconds as it raised ConnectError: [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate in certificate chain (_ssl.c:1028).
Retrying llama_cloud_services.parse.utils.make_api_request.<locals>._make_request in 4.0 seconds as it raised ConnectError: [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate in certi

Error while parsing the file '/Users/joo/heng-joo-capstone/Datasets/Graph_Dataset/generate_questions_copy.pdf': RetryError[<Future at 0x31a85bc50 state=finished raised ConnectError>]


Parsing files: 100%|██████████| 2/2 [00:24<00:00, 12.10s/it]

Error while parsing the file '/Users/joo/heng-joo-capstone/Datasets/Graph_Dataset/htx_ar2023_fa_240905.pdf': RetryError[<Future at 0x31a8b1550 state=finished raised ConnectError>]
  -> Successfully parsed 2 PDF file(s).

- Parsing other files with SimpleDirectoryReader...


  -> Successfully parsed 2 other file(s).

--- Ingestion complete ---
Successfully loaded and chunked a total of 2 document(s) from all files in '/Users/joo/heng-joo-capstone/Datasets/Graph_Dataset'.


In [4]:
# Create a node parser with overlapping chunks
node_parser = SentenceSplitter(
    chunk_size=512, 
    chunk_overlap=20
)

# Get nodes from the LlamaParse documents
nodes = node_parser.get_nodes_from_documents(documents)

print(f"📄 Split documents into {len(nodes)} nodes using SentenceSplitter.")


📄 Split documents into 15 nodes using SentenceSplitter.


## Llama 3.1 8B Instruct

In [6]:
model_name = "meta-llama/Meta-Llama-3.1-8B-Instruct"

# Initialize the tokenizer to get the token ID for our stop sequence
tokenizer = AutoTokenizer.from_pretrained(model_name, token=hf_token)
# The semicolon is our desired stop character. Get its token ID.
semicolon_token_id = tokenizer.convert_tokens_to_ids(";")

# Now, initialize the LLM with the correct stop condition
llm = HuggingFaceLLM(
    model_name=model_name,
    tokenizer_name=model_name,
    device_map="auto",
    model_kwargs={"token": hf_token, "torch_dtype": torch.bfloat16},
    # Use 'eos_token_id' which is the correct parameter for this purpose
    generate_kwargs={
        "temperature": 0.1,
        "do_sample": True,
        # This tells the model to stop generating as soon as it outputs a semicolon
        "eos_token_id": semicolon_token_id,
    }
)

print("HuggingFaceLLM initialized with the ';' character as the end-of-sequence token.")

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

2025-09-03 13:17:28,911 - WARNING - Some parameters are on the meta device because they were offloaded to the disk.


HuggingFaceLLM initialized with the ';' character as the end-of-sequence token.


## OpenAI gpt model (Entity & Relationship Extraction for Graph datastore)

In [7]:
openai_llm = OpenAI(
    api_key=openai_api_key,
    model="gpt-4o",
    temperature=0.0,
    # Uncomment to set a timeout
    # timeout=120.0,
)

print("OpenAI LLM initialized successfully.")

OpenAI LLM initialized successfully.


## Entity & Relationship Extraction with gpt

In [8]:
kg_extraction_prompt_str = """
You are an expert data extraction algorithm. Your task is to extract a knowledge graph from the provided text.
The knowledge graph should consist of nodes and relationships.

**Instructions:**
1.  **Nodes:** Identify all relevant entities and assign them one of the following labels:
    - {allowed_nodes}

2.  **Relationships:** Identify the relationships between these entities. The relationship triplets MUST follow this format: `(Head Entity, RELATIONSHIP_TYPE, Tail Entity)`.
    - The relationship type MUST be one of the following:
    - {allowed_relationships_str}

3.  **Output Format:** Your response MUST be ONLY a single, valid JSON list of triplets, where each triplet is a list of three strings: `["head_entity", "relationship_type", "tail_entity"]`.
    - Do not include any explanations, introductory text, or markdown formatting.
    - If no relationships are found, return an empty list `[]`.

**Example:**
Text: "HTX signed an agreement with Microsoft in Redmond."
JSON Output:
[["HTX", "SIGNED_AGREEMENT_WITH", "Microsoft"], ["Microsoft", "LOCATED_IN", "Redmond"]]

---
**Text to Analyze:**
{text}
---
**JSON Output:**
"""

# Define your schema
allowed_nodes = [
    "Organization", "Person", "Location", "Project", "Grouping",
    "PartnerCategory", "Concept", "Value", "Description", "Technology"
]

# Use the three-tuple format to define relationship schema
allowed_relationships = [
    ("Organization", "HAS_GROUPING", "Grouping"),
    ("Grouping", "CONTAINS_CATEGORY", "PartnerCategory"),
    ("PartnerCategory", "INCLUDES_PARTNER", "Organization"),
    # ... add all other relationship tuples from your manual graph here
]

# Format the allowed relationships for the prompt
allowed_relationships_str = "\n".join([f"- {rel[1]} (Connects {rel[0]} to {rel[2]})" for rel in allowed_relationships])

# Create the LlamaIndex PromptTemplate
kg_extraction_prompt = PromptTemplate(
    kg_extraction_prompt_str,
    prompt_type="knowledge_graph_extraction",
)

print("✅ Schema-enforcing prompt for LlamaIndex is ready.")

✅ Schema-enforcing prompt for LlamaIndex is ready.


## Graph Helper Functions

In [9]:
def filter_triplets(triplets: list, allowed_nodes: list, allowed_relationships: list) -> list:
    """
    Filters extracted triplets to ensure they conform to the predefined schema.
    This acts as the "strict_mode=True" for LlamaIndex.
    """
    # This function is not needed for the LlamaIndex KnowledgeGraphIndex as it handles this internally
    # when you provide the allowed_nodes and allowed_relationships directly.
    # However, it's a good practice to have it for custom RAG pipelines.
    # For this specific implementation, we will rely on the index's internal validation.
    # Kept here for educational purposes.
    return triplets


## Manual Ingestion into Neo4j function

In [10]:
# Helper function to create a relationship between two nodes.
def create_manual_relationship(driver, subject_label, subject_name, rel_type, object_label, object_name):
    """
    Creates two nodes and a relationship between them using explicit data.
    It uses MERGE to avoid creating duplicate nodes or relationships.
    """
    with driver.session() as session:
        # This Cypher query is robust. It finds or creates the subject and object nodes,
        # then finds or creates the relationship between them.
        query = f"""
        MERGE (s:{subject_label} {{name: $subject_name}})
        MERGE (o:{object_label} {{name: $object_name}})
        MERGE (s)-[:`{rel_type}`]->(o)
        """
        session.run(query, subject_name=subject_name, object_name=object_name)
    print(f"  - Created/Verified: ({subject_name})-[{rel_type}]->({object_name})")

## Manual Creation of Knowledge Graph

In [11]:
# Clear DB before ingesting additional relations
driver = GraphDatabase.driver(URI, auth=AUTH)
with driver.session() as session:
    session.run("MATCH (n) DETACH DELETE n")
print("Database cleared.")

# Create the root HTX node and partner category
create_manual_relationship(
    driver,
    subject_label="Root", subject_name="HTX Annual Report 2023",
    rel_type="HAS_GROUPING",
    object_label="Grouping", object_name="Partners"
)

all_partners = {
    "Strategic Partners for Innovation": ["accenture", "Microsoft", "IDEMIA", "NCS", "Singtel", "Thales", "ENSIGN", "KLASS", "NEC", "PCSS", "ROHDE&SCHWARZ", "novel", "SHIMADZU", "ST Engineering", "SEKISUI", "veredus"],
    "Academic Partners": ["MIT", "A*STAR", "Fraunhofer", "IEEE CSAIL", "NTU", "NUS", "SIT", "SUTD", "Singapore Polytechnic"],
    "Local Government Agencies": ["CSA", "DSO", "DSTA", "GOVTECH", "HSA", "IMDA", "LTA", "MINDEF", "MHA", "MOM", "NEA", "NATIONAL PARKS"],
    "Foreign Government Agencies": ["Australian Border Force", "AFP", "Ministry of National Security", "French Ministry of the Interior", "Dutch Ministry of Justice and Security", "US Department of Homeland Security", "KOREAN NATIONAL POLICE AGENCY"],
    "Other Key Industry Partners": ["aws", "CERTIS", "CISCO", "SAMSUNG", "SANS", "sats", "SOSA"]
}

# C. Loop through the data to build the graph
for category, partners in all_partners.items():
    # Link the category to the main 'Partners' grouping
    create_manual_relationship(
        driver,
        subject_label="Grouping", subject_name="Partners",
        rel_type="CONTAINS_CATEGORY",
        object_label="PartnerCategory", object_name=category
    )
    
    # Link each partner to its category
    for partner in partners:
        create_manual_relationship(
            driver,
            subject_label="PartnerCategory", subject_name=category,
            rel_type="INCLUDES_PARTNER",
            object_label="Organization", object_name=partner
        )

# Create the root HTX node and who we are category
# HTX Annual Report 2023 -> Who We Are
create_manual_relationship(
    driver,
    subject_label="Root", subject_name="HTX Annual Report 2023",
    rel_type="HAS_GROUPING",
    object_label="Grouping", object_name="Who We Are"
)

# HTX Annual Report 2023 -> Why We Exist
create_manual_relationship(
    driver,
    subject_label="Root", subject_name="HTX Annual Report 2023",
    rel_type="HAS_GROUPING",
    object_label="Grouping", object_name="Why We Exist"
)

# HTX Annual Report 2023 -> What We Do
create_manual_relationship(
    driver,
    subject_label="Root", subject_name="HTX Annual Report 2023",
    rel_type="HAS_GROUPING",
    object_label="Grouping", object_name="What We Do"
)

# --- What We Do category ---
create_manual_relationship(
    driver,
    subject_label="Grouping", subject_name="What We Do",
    rel_type="CONTAINS_CAPABILITY",
    object_label="Concept", object_name="OUR SCIENCE & TECH CAPABILITIES"
)

# Our Science & Tech Capabilities -> Digital
create_manual_relationship(
    driver,
    subject_label="Concept", subject_name="OUR SCIENCE & TECH CAPABILITIES",
    rel_type="CONTAINS_AREA",
    object_label="Concept", object_name="Digital"
)

# Our Science & Tech Capabilities -> Engineering
create_manual_relationship(
    driver,
    subject_label="Concept", subject_name="OUR SCIENCE & TECH CAPABILITIES",
    rel_type="CONTAINS_AREA",
    object_label="Concept", object_name="Engineering"
)

# Our Science & Tech Capabilities -> Sciences
create_manual_relationship(
    driver,
    subject_label="Concept", subject_name="OUR SCIENCE & TECH CAPABILITIES",
    rel_type="CONTAINS_AREA",
    object_label="Concept", object_name="Sciences"
)

# Digital -> 

# HTX Annual Report 2023 -> Board of Directors
create_manual_relationship(
    driver,
    subject_label="Root", subject_name="HTX Annual Report 2023",
    rel_type="HAS_GROUPING",
    object_label="Grouping", object_name="Board of Directors"
)

# HTX Annual Report 2023 -> Senior Management
create_manual_relationship(
    driver,
    subject_label="Root", subject_name="HTX Annual Report 2023",
    rel_type="HAS_GROUPING",
    object_label="Grouping", object_name="Senior Management"
)

# --- Why We Exist category ---

# Why We Exist -> Mission
create_manual_relationship(
    driver,
    subject_label="Grouping", subject_name="Why We Exist",
    rel_type="CONTAINS_MISSION",
    object_label="Concept", object_name="Mission"
)

# Mission -> Description
create_manual_relationship(
    driver,
    subject_label="Concept", subject_name="Mission",
    rel_type="HAS_DESCRIPTION",
    object_label="Description", object_name="Advance Science & Technology"
)

create_manual_relationship(
    driver,
    subject_label="Concept", subject_name="Mission",
    rel_type="HAS_DESCRIPTION",
    object_label="Description", object_name="Force Multiply our Home Team"
)

create_manual_relationship(
    driver,
    subject_label="Concept", subject_name="Mission",
    rel_type="HAS_DESCRIPTION",
    object_label="Description", object_name="Secure Singapore’s Future"
)

# --- Board of Directors category ---

# Chairman
create_manual_relationship(
    driver,
    subject_label="Grouping", subject_name="Board of Directors",
    rel_type="CONTAINS_MEMBER",
    object_label="Chairman", object_name="Mr Aubeck KAM Tse Tsuen"
)

# Chief Executive
create_manual_relationship(
    driver,
    subject_label="Grouping", subject_name="Board of Directors",
    rel_type="CONTAINS_MEMBER",
    object_label="Chief_Executive", object_name="Mr CHAN Tsan"
)

# SPF Commissioner
create_manual_relationship(
    driver,
    subject_label="Grouping", subject_name="Board of Directors",
    rel_type="CONTAINS_MEMBER",
    object_label="Commissioner", object_name="Mr HOONG Wee Teck"
)

# SCDF Commissioner
create_manual_relationship(
    driver,
    subject_label="Grouping", subject_name="Board of Directors",
    rel_type="CONTAINS_MEMBER",
    object_label="Commissioner", object_name="Mr YAP Wee Teck Eric"
)

# ICA Commissioner
create_manual_relationship(
    driver,
    subject_label="Grouping", subject_name="Board of Directors",
    rel_type="CONTAINS_MEMBER",
    object_label="Commissioner", object_name="Mr SIM Wai Meng Marvin"
)

# SPS Commissioner
create_manual_relationship(
    driver,
    subject_label="Grouping", subject_name="Board of Directors",
    rel_type="CONTAINS_MEMBER",
    object_label="Commissioner", object_name="Ms SHIE Yong Lee"
)

# --- Other Members ---
create_manual_relationship(
    driver,
    subject_label="Grouping", subject_name="Board of Directors",
    rel_type="CONTAINS_MEMBER",
    object_label="Member", object_name="Mr TEE Chong Fui"
)

create_manual_relationship(
    driver,
    subject_label="Grouping", subject_name="Board of Directors",
    rel_type="CONTAINS_MEMBER",
    object_label="Member", object_name="Ms Janet ANG Guat Har"
)

create_manual_relationship(
    driver,
    subject_label="Grouping", subject_name="Board of Directors",
    rel_type="CONTAINS_MEMBER",
    object_label="Member", object_name="Mr ONG Pang Thye"
)

create_manual_relationship(
    driver,
    subject_label="Grouping", subject_name="Board of Directors",
    rel_type="CONTAINS_MEMBER",
    object_label="Member", object_name="Prof CHONG Tow Chong"
)

create_manual_relationship(
    driver,
    subject_label="Grouping", subject_name="Board of Directors",
    rel_type="CONTAINS_MEMBER",
    object_label="Member", object_name="Ms Carmen WEE Yik Cheng"
)

create_manual_relationship(
    driver,
    subject_label="Grouping", subject_name="Board of Directors",
    rel_type="CONTAINS_MEMBER",
    object_label="Member", object_name="Mr Richard KOH Chin Kiong"
)

create_manual_relationship(
    driver,
    subject_label="Grouping", subject_name="Board of Directors",
    rel_type="CONTAINS_MEMBER",
    object_label="Member", object_name="Mr CHANG Yew Kong"
)

create_manual_relationship(
    driver,
    subject_label="Grouping", subject_name="Board of Directors",
    rel_type="CONTAINS_MEMBER",
    object_label="Member", object_name="Mr THAM Kok Leong"
)

create_manual_relationship(
    driver,
    subject_label="Grouping", subject_name="Board of Directors",
    rel_type="CONTAINS_MEMBER",
    object_label="Member", object_name="Ms CHEW Seow-Chien"
)

create_manual_relationship(
    driver,
    subject_label="Grouping", subject_name="Board of Directors",
    rel_type="CONTAINS_MEMBER",
    object_label="Member", object_name="Ms Gwenda FONG Su-Yi"
)

create_manual_relationship(
    driver,
    subject_label="Grouping", subject_name="Board of Directors",
    rel_type="CONTAINS_MEMBER",
    object_label="Member", object_name="Prof LUI Pao Chuen"
)

create_manual_relationship(
    driver,
    subject_label="Grouping", subject_name="Board of Directors",
    rel_type="CONTAINS_MEMBER",
    object_label="Member", object_name="Mr Prithvi Suresh RAI"
)

# --- Senior Management category ---
create_manual_relationship(
    driver,
    subject_label="Grouping", subject_name="Senior Management",
    rel_type="CONTAINS_MEMBER",
    object_label="Chief_Executive", object_name="Mr CHAN Tsan"
)

create_manual_relationship(
    driver,
    subject_label="Deputy_Chief_Executive", subject_name="Mr CHEN Yeang Tat",
    rel_type="REPORTS_TO",
    object_label="Chief_Executive", object_name="Mr CHAN Tsan"
)

# Chief Executive -> Assistant Chief Executive
create_manual_relationship(
    driver,
    subject_label="Deputy_Chief_Executive", subject_name="Mr NG Yeow Boon",
    rel_type="REPORTS_TO",
    object_label="Chief_Executive", object_name="Mr CHAN Tsan"
)

# Assistant Chief Executive -> Deputy Chief Executive
create_manual_relationship(
    driver,
    subject_label="Assistant_Chief_Executive", subject_name="Mr ANG Chee Wee",
    rel_type="REPORTS_TO",
    object_label="Deputy_Chief_Executive", object_name="Mr NG Yeow Boon"
)

# Assistant Chief Executive -> Deputy Chief Executive
create_manual_relationship(
    driver,
    subject_label="Assistant_Chief_Executive", subject_name="Dr LIM Kia Yong",
    rel_type="REPORTS_TO",
    object_label="Deputy_Chief_Executive", object_name="Mr CHEN Yeang Tat"
)

create_manual_relationship(
    driver,
    subject_label="Assistant_Chief_Executive", subject_name="Mr Colin TAN",
    rel_type="SUPERVISES",
    object_label="Deputy_Chief_Executive", object_name="Mr CHEN Yeang Tat"
)

# --- Who We Are category ---

# Who We Are -> Our Values
create_manual_relationship(
    driver,
    subject_label="Grouping", subject_name="Who We Are",
    rel_type="CONTAINS_VALUES",
    object_label="Concept", object_name="Our Values"
)

# Who We Are -> Our Culture & People
create_manual_relationship(
    driver,
    subject_label="Grouping", subject_name="Who We Are",
    rel_type="CONTAINS_CULTURE",
    object_label="Concept", object_name="Our Culture & People"
)

# Who We Are -> Our Commitment to Sustainability
create_manual_relationship(
    driver,
    subject_label="Grouping", subject_name="Who We Are",
    rel_type="CONTAINS_COMMITMENT",
    object_label="Concept", object_name="Our Commitment to Sustainability"
)

# Who We Are -> Our Vision
create_manual_relationship(
    driver,
    subject_label="Grouping", subject_name="Who We Are",
    rel_type="CONTAINS_VISION",
    object_label="Concept", object_name="Our Vision"
)

# --- Our Vision and Description ---

# Our Vision -> Description
create_manual_relationship(
    driver,
    subject_label="Concept", subject_name="Our Vision",
    rel_type="HAS_DESCRIPTION",
    object_label="Description", object_name="Exponentially Impacting Singapore’s Safety and Security"
)

# --- Our Values concept and its values ---

# Our Values -> Mission
create_manual_relationship(
    driver,
    subject_label="Concept", subject_name="Our Values",
    rel_type="CONTAINS_VALUE",
    object_label="Value", object_name="Mission"
)

# Our Values -> Teamwork
create_manual_relationship(
    driver,
    subject_label="Concept", subject_name="Our Values",
    rel_type="CONTAINS_VALUE",
    object_label="Value", object_name="Teamwork"
)

# Our Values -> Empathy
create_manual_relationship(
    driver,
    subject_label="Concept", subject_name="Our Values",
    rel_type="CONTAINS_VALUE",
    object_label="Value", object_name="Empathy"
)

# Our Values -> Exuberance
create_manual_relationship(
    driver,
    subject_label="Concept", subject_name="Our Values",
    rel_type="CONTAINS_VALUE",
    object_label="Value", object_name="Exuberance"
)

# Our Values -> Foresight
create_manual_relationship(
    driver,
    subject_label="Concept", subject_name="Our Values",
    rel_type="CONTAINS_VALUE",
    object_label="Value", object_name="Foresight"
)

# Our Values -> Innovation
create_manual_relationship(
    driver,
    subject_label="Concept", subject_name="Our Values",
    rel_type="CONTAINS_VALUE",
    object_label="Value", object_name="Innovation"
)

# --- Value descriptions ---

# Teamwork -> Description
create_manual_relationship(
    driver,
    subject_label="Value", subject_name="Teamwork",
    rel_type="HAS_DESCRIPTION",
    object_label="Description", object_name="We work together to make the extraordinary happen"
)

# Empathy -> Description
create_manual_relationship(
    driver,
    subject_label="Value", subject_name="Empathy",
    rel_type="HAS_DESCRIPTION",
    object_label="Description", object_name="We appreciate and care for one another, and celebrate our achievements together"
)

# Mission -> Description
create_manual_relationship(
    driver,
    subject_label="Value", subject_name="Mission",
    rel_type="HAS_DESCRIPTION",
    object_label="Description", object_name="We are the Home Team’s Force Multiplier"
)

# Foresight -> Description
create_manual_relationship(
    driver,
    subject_label="Value", subject_name="Foresight",
    rel_type="HAS_DESCRIPTION",
    object_label="Description", object_name="We apply exceptional thinking to anticipate future threats and opportunities"
)

# Exuberance -> Description
create_manual_relationship(
    driver,
    subject_label="Value", subject_name="Exuberance",
    rel_type="HAS_DESCRIPTION",
    object_label="Description", object_name="We exude energy, optimism and a can-do attitude in pursuit of excellence"
)

# Innovation -> Description
create_manual_relationship(
    driver,
    subject_label="Value", subject_name="Innovation",
    rel_type="HAS_DESCRIPTION",
    object_label="Description", object_name="We constantly experiment, undaunted by failure, to create solutions for tomorrow’s challenges"
)

# --- Our Commitment to Sustainability -> Description ---
create_manual_relationship(
    driver,
    subject_label="Concept", subject_name="Our Commitment to Sustainability",
    rel_type="HAS_DESCRIPTION",
    object_label="Description", object_name="Lower Energy Utilisation Index and Water Efficiency Index by 10% by 2030"
)

create_manual_relationship(
    driver,
    subject_label="Concept", subject_name="Our Commitment to Sustainability",
    rel_type="HAS_DESCRIPTION",
    object_label="Description", object_name="Lower Waste Disposal Index by 30% by 2030"
)

create_manual_relationship(
    driver,
    subject_label="Concept", subject_name="Our Commitment to Sustainability",
    rel_type="HAS_DESCRIPTION",
    object_label="Description", object_name="Achieve net-zero emissions by 2045"
)

# --- Our Culture & People ---

# Culture & People -> eXpresso!
create_manual_relationship(
    driver,
    subject_label="Concept", subject_name="Our Culture & People",
    rel_type="CONTAINS_INITIATIVE",
    object_label="Initiative", object_name="eXpresso!"
)
# Culture & People -> Growing HTX's culture of innovation
create_manual_relationship(
    driver,
    subject_label="Concept", subject_name="Our Culture & People",
    rel_type="CONTAINS_INITIATIVE",
    object_label="Initiative", object_name="Growing HTX's culture of innovation"
)

# Culture & People -> Undaunted Award
create_manual_relationship(
    driver,
    subject_label="Concept", subject_name="Our Culture & People",
    rel_type="CONTAINS_INITIATIVE",
    object_label="Initiative", object_name="Undaunted Award"
)

# Culture & People -> Family Day
create_manual_relationship(
    driver,
    subject_label="Concept", subject_name="Our Culture & People",
    rel_type="CONTAINS_INITIATIVE",
    object_label="Initiative", object_name="Family Day"
)

# Culture & People -> HTX Annual Walk & Run
create_manual_relationship(
    driver,
    subject_label="Concept", subject_name="Our Culture & People",
    rel_type="CONTAINS_INITIATIVE",
    object_label="Initiative", object_name="HTX Annual Walk & Run"
)

# Culture & People -> HTX Annual Cycle
create_manual_relationship(
    driver,
    subject_label="Concept", subject_name="Our Culture & People",
    rel_type="CONTAINS_INITIATIVE",
    object_label="Initiative", object_name="HTX Annual Cycle"
)

# Culture & People -> Bringing the X-Factor to the Purple Parade
create_manual_relationship(
    driver,
    subject_label="Concept", subject_name="Our Culture & People",
    rel_type="CONTAINS_INITIATIVE",
    object_label="Initiative", object_name="Bringing the X-Factor to the Purple Parade"
)

# Culture & People -> National Day Awards
create_manual_relationship(
    driver,
    subject_label="Concept", subject_name="Our Culture & People",
    rel_type="CONTAINS_INITIATIVE",
    object_label="Initiative", object_name="National Day Awards"
)

#  eXpresso! -> Description
create_manual_relationship(
    driver,
    subject_label="Initiative", subject_name="eXpresso!",
    rel_type="HAS_DESCRIPTION",
    object_label="Description", object_name="HTX’s eXpresso! is a unique town hall that brings people together in a relaxed and fun setting to learn about the latest happenings in HTX and interact with senior management. These casual sessions help to foster open communication and strengthen the sense of community within HTX."
)

# Growing HTX's culture of innovation -> Description
create_manual_relationship(
    driver,
    subject_label="Initiative", subject_name="Growing HTX's culture of innovation",
    rel_type="HAS_DESCRIPTION",
    object_label="Description", object_name="One of the secrets behind HTX’s quick growth over the past five years is our culture of openness and sharing, and this was reflected in the various events that HTX’s innovation lab TIGER held in partnership with industry veterans to take Xponents behind the scenes of the innovation process."
)

# Undaunted Award -> Description
create_manual_relationship(
    driver,
    subject_label="Initiative", subject_name="Undaunted Award",
    rel_type="HAS_DESCRIPTION",
    object_label="Description", object_name="This award is presented to Xponents who demonstrated that they were undaunted in the face of failure and were quick at learning from mistakes to better their innovations."
)

# Family Day -> Description
create_manual_relationship(
    driver,
    subject_label="Initiative", subject_name="Family Day",
    rel_type="HAS_DESCRIPTION",
    object_label="Description", object_name="At HTX, we play just as hard as we work, and this was reflected in our inaugural Xponents’ Family Day Out carnival. The event gave Xponents the chance to relax and bond with their colleagues and family members over a variety of activities, including carnival rides, arcade games, and performances."
)

# HTX Annual Walk & Run -> Description
create_manual_relationship(
    driver,
    subject_label="Initiative", subject_name="HTX Annual Walk & Run",
    rel_type="HAS_DESCRIPTION",
    object_label="Description", object_name="Aptly themed “Engineering Good Health”, HTX’s Annual Walk & Run saw some Xponents set off from various locations around Marina Bay and East Coast before converging at the Singapore Sports Hub for the official flag-off event."
)

# HTX Annual Cycle -> Description
create_manual_relationship(
    driver,
    subject_label="Initiative", subject_name="HTX Annual Cycle",
    rel_type="HAS_DESCRIPTION",
    object_label="Description", object_name="The HTX’s Annual Cycle took participants on an idyllic course stretching from the Road Safety Park to the National Sailing Centre and Marina East Park and back. The event transitioned into a Virtual Cycling race allowing teams to cycle at their own pace and record mileage with apps."
)

# Bringing the X-Factor to the Purple Parade -> Description
create_manual_relationship(
    driver,
    subject_label="Initiative", subject_name="Bringing the X-Factor to the Purple Parade",
    rel_type="HAS_DESCRIPTION",
    object_label="Description", object_name="Our Xponents don’t just have big minds. They have big hearts too. One person who exemplifies this is Director of Platform Systems Sustainment Centre Tan Teck Chuan. An avid balloon sculptor, Teck Chuan has been crafting balloons to support charitable causes for over a decade. He and a team of kind-hearted balloon sculptors participated in the Purple Parade to support charities for persons with disabilities. All proceeds from their balloon sales were donated to the cause. Additionally, a contingent of Xponents from the Building and Infrastructure (B&I) Sustainment Centre also participated in the Purple Parade and helped raise awareness and support of the event. The combined efforts of the balloon sculpture team and the B&I contingent resulted in over S$6,000 raised for charity – the highest the balloon sculpture team raised in over a decade. The balloon sculptures proved to be a big hit at the Purple Parade, and Teck Chuan’s team continues to teach this craft at community centers across Singapore."
)

# National Day Awards -> Description
create_manual_relationship(
    driver,
    subject_label="Initiative", subject_name="National Day Awards",
    rel_type="HAS_DESCRIPTION",
    object_label="Description", object_name="Thirty-one Xponents received the Public Administration Medal, Commendation Medal, Efficiency Medal and Long Service Medal. Kudos to all of the awardees!"
)

print("\n--- Manual knowledge graph creation complete. ---")

driver.close()

Database cleared.
  - Created/Verified: (HTX Annual Report 2023)-[HAS_GROUPING]->(Partners)
  - Created/Verified: (Partners)-[CONTAINS_CATEGORY]->(Strategic Partners for Innovation)
  - Created/Verified: (Strategic Partners for Innovation)-[INCLUDES_PARTNER]->(accenture)
  - Created/Verified: (Strategic Partners for Innovation)-[INCLUDES_PARTNER]->(Microsoft)
  - Created/Verified: (Strategic Partners for Innovation)-[INCLUDES_PARTNER]->(IDEMIA)
  - Created/Verified: (Strategic Partners for Innovation)-[INCLUDES_PARTNER]->(NCS)
  - Created/Verified: (Strategic Partners for Innovation)-[INCLUDES_PARTNER]->(Singtel)
  - Created/Verified: (Strategic Partners for Innovation)-[INCLUDES_PARTNER]->(Thales)
  - Created/Verified: (Strategic Partners for Innovation)-[INCLUDES_PARTNER]->(ENSIGN)
  - Created/Verified: (Strategic Partners for Innovation)-[INCLUDES_PARTNER]->(KLASS)
  - Created/Verified: (Strategic Partners for Innovation)-[INCLUDES_PARTNER]->(NEC)
  - Created/Verified: (Strategic Pa

## Graph Query Engine

In [ ]:
graph_store = Neo4jGraphStore(
    url=URI,
    username=os.getenv("NEO4J_DATABASE"), # The username from your .env file
    password=os.getenv("NEO4J_PASSWORD"), # The password from your .env file
    database="neo4j"                      # Explicitly define the database name
)

storage_context = StorageContext.from_defaults(graph_store=graph_store)

# 2. Build the KnowledgeGraphIndex with your schema
# The index constructor will extract triplets using your custom prompt and store them.
# kg_index = KnowledgeGraphIndex(
#     nodes,  # Use the nodes created by SentenceSplitter
#     storage_context=storage_context,
#     kg_extraction_prompt_template=kg_extraction_prompt,
#     llm=openai_llm,  # Use your powerful GPT-4o for extraction
#     max_triplets_per_chunk=15,
#     include_embeddings=True,  # Recommended for hybrid search later
#     # Pass the schema for LlamaIndex's internal validation (acts like strict_mode)
#     allowed_kg_nodes=allowed_nodes,
#     allowed_kg_relationships=[r[1] for r in allowed_relationships]
# )

# print("✅ KnowledgeGraphIndex built and data ingested into Neo4j.")

# Prompt template to generate cypher query
DEFAULT_KG_QUERY_SYNTHESIS_TMPL = (
"You are an expert Cypher query generator. Your sole task is to generate a single, "
"syntactically correct Cypher query to answer the user's question based on the provided graph schema. "
"Do not provide any explanations, introductory text, or markdown formatting. "
"Your response MUST be ONLY the raw Cypher query and nothing else. "
"Start your response directly with a Cypher keyword like 'MATCH' or 'OPTIONAL MATCH'.\n\n"
"Schema:\n"
"---------------------\n"
"{schema}\n"
"---------------------\n"
"User's Question: {query_str}\n"
"Cypher Query:"
)

# Prompt template to generate response
DEFAULT_RESPONSE_SYNTHESIS_TMPL = (
    "You are a helpful assistant. You have been provided with the results of a Cypher query "
    "from a knowledge graph and the original user question. "
    "Synthesize a conversational answer based on the provided information. "
    "Do not mention Cypher or the knowledge graph in your response.\n"
    "User question: {query_str}\n"
    "Query results: {context_str}\n"
    "Answer: "
)

kg_query_synthesis_prompt = PromptTemplate(DEFAULT_KG_QUERY_SYNTHESIS_TMPL)
kg_response_answer_prompt = PromptTemplate(DEFAULT_RESPONSE_SYNTHESIS_TMPL)


# Initialize the query engine with the storage_context
graph_query_engine = KnowledgeGraphQueryEngine(
    storage_context=storage_context,
    graph_query_synthesis_prompt=kg_query_synthesis_prompt,
    graph_response_answer_prompt=kg_response_answer_prompt,
    verbose=True # Check Cypher Query
)

print("Knowledge graph query engine is ready.")

/tmp/ipykernel_4373/3194465031.py:57: DeprecationWarning: Call to deprecated class KnowledgeGraphQueryEngine. (KnowledgeGraphQueryEngine is deprecated. It is recommended to use the PropertyGraphIndex and associated retrievers instead.) -- Deprecated since version 0.10.53.
  graph_query_engine = KnowledgeGraphQueryEngine(


Knowledge graph query engine is ready.


## Query Graph Datastore

In [ ]:
# Define question for graph
query_text_graph = "List down HTX strategic partners?"

# Query the engine
graph_response = graph_query_engine.query(query_text_graph)

# Print the response
print(str(graph_response))

Graph Store Query:
MATCH (:Root)-[:HAS_GROUPING]->(:Grouping)-[:CONTAINS_CATEGORY]->(:PartnerCategory)-[:INCLUDES_PARTNER]->(partner:Organization)
RETURN partner.name
Graph Store Response:
[{'partner.name': 'aws'}, {'partner.name': 'CERTIS'}, {'partner.name': 'CISCO'}, {'partner.name': 'SAMSUNG'}, {'partner.name': 'SANS'}, {'partner.name': 'sats'}, {'partner.name': 'SOSA'}, {'partner.name': 'accenture'}, {'partner.name': 'Microsoft'}, {'partner.name': 'IDEMIA'}, {'partner.name': 'NCS'}, {'partner.name': 'Singtel'}, {'partner.name': 'Thales'}, {'partner.name': 'ENSIGN'}, {'partner.name': 'KLASS'}, {'partner.name': 'NEC'}, {'partner.name': 'PCSS'}, {'partner.name': 'ROHDE&SCHWARZ'}, {'partner.name': 'novel'}, {'partner.name': 'SHIMADZU'}, {'partner.name': 'ST Engineering'}, {'partner.name': 'SEKISUI'}, {'partner.name': 'veredus'}, {'partner.name': 'MIT'}, {'partner.name': 'A*STAR'}, {'partner.name': 'Fraunhofer'}, {'partner.name': 'IEEE CSAIL'}, {'partner.name': 'NTU'}, {'partner.name': 